# HAMBEE EVA4 · 714 imágenes originales

Ejecuta la celda y carga `EVA4_RHAGUER_YMALDONADO_714.zip` si aparece el selector. El modelo ONNX 1.2.0 ya está entrenado. La celda instala las dependencias, verifica FastAPI, procesa las 714 imágenes y descarga un ZIP con el CSV, los gráficos y las mediciones de la sesión.

In [ ]:
import csv
import json
import os
from pathlib import Path
import shutil
import stat
import subprocess
import sys
from tempfile import mkdtemp
from zipfile import ZipFile

from IPython.display import FileLink, Markdown, display

ZIP_NAME = "EVA4_RHAGUER_YMALDONADO_714.zip"
bundle = Path(ZIP_NAME).resolve()
if not bundle.is_file():
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if Path(name).suffix.lower() == ".zip"]
    if len(zip_names) != 1:
        raise ValueError("Selecciona un único ZIP de la entrega HAMBEE de 714 imágenes.")
    bundle = Path(zip_names[0]).resolve()

work = Path(mkdtemp(prefix="hambee-714-", dir=Path.cwd()))
with ZipFile(bundle) as archive:
    entries = archive.infolist()
    paths = set()
    extract_entries = []
    total_size = 0
    for entry in entries:
        target = (work / entry.filename).resolve()
        if (not target.is_relative_to(work) or target == work or target in paths
                or "\\" in entry.filename or stat.S_ISLNK(entry.external_attr >> 16)):
            raise ValueError(f"Entrada ZIP inválida: {entry.filename}")
        paths.add(target)
        if any(part == ".venv" or part.startswith(".venv-")
               for part in entry.filename.split("/")):
            continue
        extract_entries.append(entry)
        total_size += entry.file_size
    if len(extract_entries) > 1000 or total_size > 200_000_000:
        raise ValueError("El tamaño o la estructura no corresponde a la entrega HAMBEE.")
    for entry in extract_entries:
        archive.extract(entry, work)

project = work / "EVA4"
dataset = "data/Dataset.zip"
model = json.loads((project / "models/metadata.json").read_text(encoding="utf-8"))
if model["counts"] != {"train": 571, "val": 72, "test": 71} or model["model_version"] != "1.2.0":
    raise ValueError("El ZIP cargado no corresponde a la entrega v1.2.0 de 714 imágenes.")

print("Instalando dependencias de inferencia y gráficos en un entorno aislado…")
environment = work / ".venv"

# Instalamos y usamos virtualenv en lugar de venv que falla en Colab
subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv", "-q"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", str(environment), "-q"], check=True)

python = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
subprocess.run([str(python), "-m", "pip", "install", "--disable-pip-version-check", "-q",
                "-r", "requirements-test.txt", "matplotlib==3.10.8"], cwd=project,
               check=True, timeout=900)

results = work / "resultados"
results.mkdir()

def run(script, *args):
    command = [str(python), script, *map(str, args)]
    child_env = None
    if script == "graficos.py":
        child_env = os.environ.copy()
        child_env["MPLBACKEND"] = "Agg"
    completed = subprocess.run(command, cwd=project, text=True, encoding="utf-8",
                               capture_output=True, timeout=300, env=child_env)
    if completed.returncode:
        details = "\n".join(part for part in
                            (completed.stdout.strip(), completed.stderr.strip()) if part)
        raise RuntimeError(f"{script} terminó con código {completed.returncode}:\n{details}")
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    return completed.stdout

print("Validando el dataset y el servicio FastAPI…")
integrity = json.loads(run("verificar_dataset.py", dataset))
(results / "dataset.json").write_text(json.dumps(integrity, indent=2), encoding="utf-8")
contract = json.loads(run("smoke.py"))
(results / "api.json").write_text(json.dumps(contract, indent=2), encoding="utf-8")

print("Procesando 714 imágenes y midiendo rendimiento en CPU…")
full = json.loads(run("procesar_dataset.py", dataset, "--output", results))
metrics = json.loads(run("benchmark.py", dataset, "--output", results / "benchmark.json"))
run("quality_gate.py", results / "benchmark.json")
run("benchmark_lotes.py", dataset, "--output", results / "lotes-cpu.json")
opt = metrics["optimizado"]
confusion = full["test"]["confusion"]
if [[confusion[real][pred] for pred in ("NP", "P")] for real in ("NP", "P")] != opt["confusion_np_p"]:
    raise ValueError("La matriz de la predicción por lotes difiere de la prueba unitaria.")
csv_path = results / "predicciones-dataset.csv"
with csv_path.open(encoding="utf-8-sig", newline="") as stream:
    rows = list(csv.DictReader(stream))
if full["processed"] != 714 or len(rows) != 714 or len({row["ruta"] for row in rows}) != 714:
    raise ValueError("El procesamiento debe producir 714 predicciones únicas.")

run("graficos.py", results, "--output", results / "HAMBEE_EVA4_RESULTADOS.png")
from IPython.display import Image
display(Markdown(
    f"### HAMBEE {model['model_version']} · época {model['best_epoch']}/{len(model['history'])}\n"
    f"- Prueba reservada: **{full['test']['correct']}/71** ({opt['accuracy']:.2%}) · F1 macro {opt['macro_f1']:.4f}\n"
    f"- Dataset: **714 originales** en {full['processing_seconds']:.2f} s.\n"
    f"- Latencia p95 del modelo optimizado: {opt['p95_ms']:.2f} ms en este equipo.\n"
    f"- FastAPI: HTTP {', '.join(map(str, contract['validated_status_codes']))}."
))
display(Image(filename=str(results / "HAMBEE_EVA4_RESULTADOS.png")))
archive_path = Path(shutil.make_archive(str(work / "HAMBEE_EVA4_RESULTADOS"), "zip", results))
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(csv_path.relative_to(Path.cwd()))))
    display(FileLink(str(archive_path.relative_to(Path.cwd()))))
else:
    files.download(str(archive_path))

